## Junção das bases de dados

Nesse notebook é feita a junção das bases de dados e a configuração final para a análise

In [1]:
import pandas as pd
import geopandas as gpd
import unicodedata
import re
from utils.load_csv import load_csv
from utils.load_shp import load_shp
from utils.save_shp import save_shp

In [2]:
votos_local = load_csv("votos_local.csv")

locais_votacao = load_csv("locais_votacao.csv")

pbf_distrito = load_csv("pbf_distrito.csv")

distritos_filtrados = load_shp("distritos_filtrado.shp")

In [3]:
def padronizar_nome(nome):

    nome = str(nome).strip().lower()

    nome = unicodedata.normalize("NFKD", nome)
    nome = "".join(
        caractere for caractere in nome
        if not unicodedata.combining(caractere)
    )

    nome = re.sub(r"[^a-z0-9]", "", nome)

    return nome

votos_local["nr_zona"] = pd.to_numeric(
    votos_local["nr_zona"], errors="coerce"
)

votos_local["nr_local"] = pd.to_numeric(
    votos_local["nr_local"], errors="coerce"
)

locais_votacao["zona"] = pd.to_numeric(
    locais_votacao["zona"], errors="coerce"
)

locais_votacao["nr_local"] = pd.to_numeric(
    locais_votacao["nr_local"], errors="coerce"
)

pbf_distrito["chave_distrito"] = (
    pbf_distrito["distrito"].apply(padronizar_nome)
)

distritos_filtrados["chave_distrito"] = (
    distritos_filtrados["NM_DIST"].apply(padronizar_nome)
)

In [4]:
duplicados = votos_local.duplicated(
    subset=["nr_zona", "nr_local"],
    keep=False
)

votos_ambiguos = votos_local[duplicados].copy()

votos_unicos = votos_local[~duplicados].copy()

votos_georreferenciados = votos_unicos.merge(
    locais_votacao[
        [
            "zona",
            "nr_local",
            "id_local",
            "nome_local",
            "endereco",
            "lat",
            "lon"
        ]
    ],
    left_on=["nr_zona", "nr_local"],
    right_on=["zona", "nr_local"],
    how="left",
    validate="many_to_one"
)

votos_sem_coordenadas = votos_georreferenciados[
    votos_georreferenciados["lat"].isna() |
    votos_georreferenciados["lon"].isna()
].copy()

votos_georreferenciados = votos_georreferenciados.dropna(
    subset=["lat", "lon"]
).copy()

gdf_votos = gpd.GeoDataFrame(
    votos_georreferenciados,
    geometry=gpd.points_from_xy(
        votos_georreferenciados["lon"],
        votos_georreferenciados["lat"]
    ),
    crs="EPSG:4326"
)

gdf_votos = gdf_votos.to_crs("EPSG:4674")

gdf_votos = gpd.sjoin(
    gdf_votos,
    distritos_filtrados[
        ["CD_DIST", "NM_DIST", "chave_distrito", "geometry"]
    ],
    how="left",
    predicate="within"
)

votos_fora_distritos = gdf_votos[
    gdf_votos["NM_DIST"].isna()
].copy()

gdf_votos = gdf_votos.dropna(
    subset=["NM_DIST"]
).copy()

In [5]:
votacao_distrito = (
    gdf_votos
    .groupby(["CD_DIST", "NM_DIST"], as_index=False)
    .agg(
        vt_val_flavio=("vt_val_flavio", "sum"),
        vt_val_lula=("vt_val_lula", "sum"),
        total_vt_val=("total_vt_val", "sum"),
        n_locais_votacao=("id_local", "nunique")
    )
)

votacao_distrito["pct_val_flavio"] = (
    votacao_distrito["vt_val_flavio"] /
    votacao_distrito["total_vt_val"] * 100
)

votacao_distrito["pct_val_lula"] = (
    votacao_distrito["vt_val_lula"] /
    votacao_distrito["total_vt_val"] * 100
)

In [6]:
gdf_locais = gpd.GeoDataFrame(
    locais_votacao.dropna(subset=["lat", "lon"]).copy(),
    geometry=gpd.points_from_xy(
        locais_votacao.dropna(subset=["lat", "lon"])["lon"],
        locais_votacao.dropna(subset=["lat", "lon"])["lat"]
    ),
    crs="EPSG:4326"
)

gdf_locais = gdf_locais.to_crs("EPSG:4674")

gdf_locais = gpd.sjoin(
    gdf_locais,
    distritos_filtrados[
        ["CD_DIST", "NM_DIST", "geometry"]
    ],
    how="left",
    predicate="within"
)

estrutura_distrito = (
    gdf_locais
    .dropna(subset=["NM_DIST"])
    .groupby(["CD_DIST", "NM_DIST"], as_index=False)
    .agg(
        n_locais=("id_local", "nunique"),
        qt_secoes=("qt_secoes", "sum"),
        qt_eleitores=("qt_eleitores", "sum")
    )
)

In [7]:
pbf_final = pbf_distrito[
    [
        "chave_distrito",
        "familias_pbf",
        "pop_total",
        "pbf_mil_hab"
    ]
].copy()

assert pbf_final["chave_distrito"].is_unique

assert distritos_filtrados["chave_distrito"].is_unique

In [8]:
base_distritos = distritos_filtrados[
    ["CD_DIST", "NM_DIST", "chave_distrito", "geometry"]
].copy()

base_distritos = base_distritos.merge(
    votacao_distrito.drop(columns=["NM_DIST"]),
    on="CD_DIST",
    how="left",
    validate="one_to_one"
)

base_distritos = base_distritos.rename(
    columns={
        "n_locais_votacao": "n_locais_com_resultado"
    }
)

base_distritos = base_distritos.merge(
    estrutura_distrito.drop(columns=["NM_DIST"]),
    on="CD_DIST",
    how="left",
    validate="one_to_one"
)

base_distritos = base_distritos.merge(
    pbf_final,
    on="chave_distrito",
    how="left",
    validate="one_to_one"
)

base_distritos = base_distritos.drop(
    columns=["chave_distrito"]
)

base_distritos = gpd.GeoDataFrame(
    base_distritos,
    geometry="geometry",
    crs=distritos_filtrados.crs
)

base_distritos = base_distritos.sort_values(
    "NM_DIST"
).reset_index(drop=True)

base_distritos.rename(
    columns={
        "CD_DIST": "cd_dist",
        "NM_DIST": "distrito"
    },
    inplace=True
)

colunas = [
    "cd_dist",
    "distrito",
    'geometry',

    "vt_val_lula",
    "vt_val_flavio",
    "total_vt_val",
    "pct_val_lula",
    "pct_val_flavio",

    "familias_pbf",
    "pop_total",
    "pbf_mil_hab",

    "n_locais",
    "n_locais_com_resultado",
    "qt_secoes",
    "qt_eleitores"
]

base_distritos = base_distritos[colunas]

base_distritos["soma_percentuais"] = (
    base_distritos["pct_val_flavio"] +
    base_distritos["pct_val_lula"]
)

In [9]:
# Conferir se todos os 96 distritos estão presentes
assert len(base_distritos) == 96

assert base_distritos["cd_dist"].is_unique

assert base_distritos["geometry"].notna().all()


# Conferir valores ausentes
print("Valores ausentes por variável:")
display(
    base_distritos.isna().sum()
)


# Conferir cobertura da votação
votos_originais = votos_local["total_vt_val"].sum()

votos_georreferenciados_total = (
    gdf_votos["total_vt_val"].sum()
)

cobertura = (
    votos_georreferenciados_total /
    votos_originais * 100
)

print(f"Votos válidos na base original: {votos_originais:,}")
print(
    f"Votos válidos georreferenciados: "
    f"{votos_georreferenciados_total:,}"
)
print(f"Cobertura da votação: {cobertura:.2f}%")

Valores ausentes por variável:


cd_dist                   0
distrito                  0
geometry                  0
vt_val_lula               0
vt_val_flavio             0
total_vt_val              0
pct_val_lula              0
pct_val_flavio            0
familias_pbf              0
pop_total                 0
pbf_mil_hab               0
n_locais                  0
n_locais_com_resultado    0
qt_secoes                 0
qt_eleitores              0
soma_percentuais          0
dtype: int64

Votos válidos na base original: 6,962,277
Votos válidos georreferenciados: 6,927,570
Cobertura da votação: 99.50%


In [11]:
#save_shp(base_distritos, "base_distritos.shp")